# 🚦 Лабораторная работа 1. Первая нейронная сеть: «Светофор»

> Практическая часть **главы 1 — «Основы нейронных сетей»** проекта **Almaz_AI**.

## 🎯 Цель

На практике пройти полный цикл работы небольшой нейронной сети:

1. подготовить данные;
2. представить их как `Tensor`;
3. создать модель;
4. посмотреть ответы до обучения;
5. обучить сеть;
6. наблюдать изменение `Loss`;
7. вычислить `Accuracy`;
8. сравнить ответы до и после обучения;
9. посмотреть обученные параметры;
10. провести самостоятельные эксперименты.

Главная цепочка:

```text
Данные → Tensor → Model → Forward Pass → Prediction
      → Loss → Backpropagation → Optimizer → Новые веса
```


# 1. Правила учебного светофора

Входы:

- красный;
- жёлтый;
- зелёный.

Кодирование:

```text
0 = выключен
1 = включён
```

Выход:

```text
0 = СТОЯТЬ
1 = ИДТИ
```

| Красный | Жёлтый | Зелёный | Решение |
|---:|---:|---:|---|
| 0 | 0 | 0 | Стоять |
| 0 | 0 | 1 | Идти |
| 0 | 1 | 0 | Стоять |
| 0 | 1 | 1 | Идти |
| 1 | 0 | 0 | Стоять |
| 1 | 0 | 1 | Стоять |
| 1 | 1 | 0 | Стоять |
| 1 | 1 | 1 | Стоять |


# 2. Импорт библиотек

Используем:

- `torch` — Tensor и обучение;
- `torch.nn` — слои нейросети;
- `pandas` — таблицы;
- `matplotlib` — графики.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

# 3. Создание датасета

In [ ]:
def create_traffic_light_dataset() -> tuple[torch.Tensor, torch.Tensor]:
    """Создаёт полный датасет состояний светофора и правильных решений."""

    features = torch.tensor(
        [
            [0.0, 0.0, 0.0],
            [0.0, 0.0, 1.0],
            [0.0, 1.0, 0.0],
            [0.0, 1.0, 1.0],
            [1.0, 0.0, 0.0],
            [1.0, 0.0, 1.0],
            [1.0, 1.0, 0.0],
            [1.0, 1.0, 1.0],
        ],
        dtype=torch.float32,
    )

    targets = torch.tensor(
        [[0.0], [1.0], [0.0], [1.0], [0.0], [0.0], [0.0], [0.0]],
        dtype=torch.float32,
    )

    return features, targets


features, targets = create_traffic_light_dataset()

print("features:", features.shape)
print("targets:", targets.shape)

In [ ]:
dataset_table = pd.DataFrame(
    {
        "Красный": features[:, 0].int().numpy(),
        "Жёлтый": features[:, 1].int().numpy(),
        "Зелёный": features[:, 2].int().numpy(),
        "Правильное решение": [
            "ИДТИ" if value == 1 else "СТОЯТЬ"
            for value in targets.squeeze().int().numpy()
        ],
    }
)

dataset_table

# 4. Создаём нейронную сеть

Архитектура:

```text
3 входа → Linear(3,4) → ReLU → Linear(4,1) → логит
```


In [ ]:
class TrafficLightNetwork(nn.Module):
    """Небольшая нейронная сеть для учебной задачи со светофором."""

    def __init__(self) -> None:
        """Создаёт два линейных слоя и ReLU."""

        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(3, 4),
            nn.ReLU(),
            nn.Linear(4, 1),
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        """Выполняет прямой проход и возвращает логиты модели."""

        return self.network(features)


model = TrafficLightNetwork()
model

# 5. Параметры модели до обучения

In [ ]:
def show_model_parameters(model: nn.Module) -> None:
    """Выводит имена, размеры и значения параметров модели."""

    for name, parameter in model.named_parameters():
        print(f"\n{name}")
        print("shape:", tuple(parameter.shape))
        print(parameter.detach())


show_model_parameters(model)

# 6. Предсказания до обучения

In [ ]:
def get_predictions(
    model: nn.Module,
    features: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Возвращает вероятности и бинарные решения модели."""

    model.eval()

    with torch.no_grad():
        logits = model(features)
        probabilities = torch.sigmoid(logits)
        predictions = (probabilities >= 0.5).int()

    return probabilities, predictions


def create_prediction_table(
    model: nn.Module,
    features: torch.Tensor,
    targets: torch.Tensor,
) -> pd.DataFrame:
    """Создаёт таблицу с вероятностями и решениями модели."""

    probabilities, predictions = get_predictions(model, features)

    return pd.DataFrame(
        {
            "Красный": features[:, 0].int().numpy(),
            "Жёлтый": features[:, 1].int().numpy(),
            "Зелёный": features[:, 2].int().numpy(),
            "Вероятность ИДТИ": probabilities.squeeze().numpy(),
            "Ответ сети": [
                "ИДТИ" if value == 1 else "СТОЯТЬ"
                for value in predictions.squeeze().numpy()
            ],
            "Правильный ответ": [
                "ИДТИ" if value == 1 else "СТОЯТЬ"
                for value in targets.squeeze().int().numpy()
            ],
        }
    )


predictions_before = create_prediction_table(model, features, targets)
predictions_before

# 7. Функция потерь и оптимизатор

In [ ]:
loss_function = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.05,
)

print(loss_function)
print(optimizer)

# 8. Обучение

На каждой эпохе:

```text
Forward Pass
→ Loss
→ zero_grad()
→ backward()
→ optimizer.step()
```


In [ ]:
def calculate_accuracy(
    model: nn.Module,
    features: torch.Tensor,
    targets: torch.Tensor,
) -> float:
    """Вычисляет долю правильных бинарных решений модели."""

    _, predictions = get_predictions(model, features)
    correct = predictions.float() == targets
    return float(correct.float().mean().item())


def train_model(
    model: nn.Module,
    features: torch.Tensor,
    targets: torch.Tensor,
    loss_function: nn.Module,
    optimizer: torch.optim.Optimizer,
    epochs: int = 1000,
) -> tuple[list[float], list[float]]:
    """Обучает модель и возвращает историю Loss и Accuracy."""

    loss_history: list[float] = []
    accuracy_history: list[float] = []

    for epoch in range(epochs):
        model.train()

        logits = model(features)
        loss = loss_function(logits, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        accuracy = calculate_accuracy(model, features, targets)

        loss_history.append(float(loss.item()))
        accuracy_history.append(accuracy)

        if epoch % 100 == 0:
            print(
                f"Epoch {epoch:04d} | "
                f"Loss={loss.item():.6f} | "
                f"Accuracy={accuracy:.2%}"
            )

    return loss_history, accuracy_history


loss_history, accuracy_history = train_model(
    model=model,
    features=features,
    targets=targets,
    loss_function=loss_function,
    optimizer=optimizer,
    epochs=1000,
)

# 9. График Loss

In [ ]:
def plot_loss_history(loss_history: list[float]) -> None:
    """Строит график изменения Loss по эпохам."""

    plt.figure(figsize=(10, 5))
    plt.plot(loss_history)
    plt.title("Обучение нейросети «Светофор»: Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.show()


plot_loss_history(loss_history)

# 10. График Accuracy

In [ ]:
def plot_accuracy_history(accuracy_history: list[float]) -> None:
    """Строит график изменения Accuracy по эпохам."""

    plt.figure(figsize=(10, 5))
    plt.plot(accuracy_history)
    plt.title("Обучение нейросети «Светофор»: Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.ylim(-0.05, 1.05)
    plt.grid(True)
    plt.show()


plot_accuracy_history(accuracy_history)

# 11. Проверка после обучения

In [ ]:
predictions_after = create_prediction_table(model, features, targets)
predictions_after

In [ ]:
final_accuracy = calculate_accuracy(model, features, targets)
print(f"Итоговая точность: {final_accuracy:.2%}")

# 12. Сравнение до и после

In [ ]:
comparison = pd.DataFrame(
    {
        "Состояние": [
            f"[{int(red)}, {int(yellow)}, {int(green)}]"
            for red, yellow, green in features.tolist()
        ],
        "До обучения": predictions_before["Ответ сети"],
        "После обучения": predictions_after["Ответ сети"],
        "Правильно": predictions_after["Правильный ответ"],
    }
)

comparison

# 13. Параметры после обучения

In [ ]:
show_model_parameters(model)

# 14. Проверка отдельного состояния

In [ ]:
def predict_traffic_light(
    model: nn.Module,
    red: int,
    yellow: int,
    green: int,
) -> tuple[float, str]:
    """Возвращает вероятность движения и текстовое решение сети."""

    state = torch.tensor(
        [[float(red), float(yellow), float(green)]],
        dtype=torch.float32,
    )

    probabilities, predictions = get_predictions(model, state)

    probability = float(probabilities.item())
    decision = "ИДТИ" if int(predictions.item()) == 1 else "СТОЯТЬ"

    return probability, decision


probability, decision = predict_traffic_light(
    model=model,
    red=0,
    yellow=1,
    green=1,
)

print(f"Вероятность ИДТИ: {probability:.4f}")
print(f"Решение: {decision}")

# 15. 🧪 Эксперименты

### Эксперимент 1 — Epochs
Попробуй `10`, `50`, `100`, `1000`.

### Эксперимент 2 — Learning Rate
Попробуй `0.001`, `0.01`, `0.05`, `0.5`.

### Эксперимент 3 — Random Seed
Меняй `torch.manual_seed(...)` до создания новой модели.

При сравнении экспериментов создавай новую модель, иначе обучение продолжится со старых весов.


# 16. ❓ Самопроверка

1. Почему `features` имеет размер `8 × 3`?
2. Почему `targets` имеет размер `8 × 1`?
3. Что находится внутри `nn.Linear`?
4. Что делает `forward()`?
5. Чем логит отличается от вероятности?
6. Что измеряет `Loss`?
7. Зачем `optimizer.zero_grad()`?
8. Что делает `loss.backward()`?
9. Что делает `optimizer.step()`?
10. Что означает одна эпоха?
11. Что показывает `Accuracy`?
12. Почему ответы до и после обучения различаются?
13. Какие параметры менялись во время обучения?
14. Почему для реального светофора лучше обычная строгая логика?


# 17. 📌 Что нужно запомнить

```text
ДАТАСЕТ
   ↓
TENSOR
   ↓
MODEL
   ↓
FORWARD PASS
   ↓
LOSS
   ↓
BACKPROPAGATION
   ↓
GRADIENTS
   ↓
OPTIMIZER
   ↓
НОВЫЕ ПАРАМЕТРЫ
   ↓
ЛУЧШЕЕ ПРЕДСКАЗАНИЕ
```

Следующая глава — **Функции активации**. В нашей модели уже присутствует `nn.ReLU()`, и дальше мы подробно разберём, зачем она нужна.
